<a href="https://www.kaggle.com/code/deepak170703/gpu-tpu-training?scriptVersionId=298006547" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import time

print("\n" + "="*70)
print("PARALLEL DEEP LEARNING WITH AUTO HARDWARE DETECTION")
print("="*70)

2026-02-16 09:56:56.141626: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771235816.332753      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771235816.389886      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771235816.841536      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771235816.841574      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771235816.841577      55 computation_placer.cc:177] computation placer alr


PARALLEL DEEP LEARNING WITH AUTO HARDWARE DETECTION


In [2]:
print("\n[STEP 1] HARDWARE DETECTION")
print("-"*70)

def detect_hardware():
    try:
        tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
        tf.config.experimental_connect_to_cluster(tpu)
        tf.tpu.experimental.initialize_tpu_system(tpu)
        strategy = tf.distribute.TPUStrategy(tpu)
        device = "TPU"
        print(f"✓ TPU DETECTED - {strategy.num_replicas_in_sync} cores")
    except:
        gpus = tf.config.list_physical_devices('GPU')
        if gpus:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            strategy = tf.distribute.MirroredStrategy()
            device = "GPU"
            print(f"✓ GPU DETECTED - {len(gpus)} device(s)")
            tf.keras.mixed_precision.set_global_policy('mixed_float16')
            print("  Mixed precision: Enabled")
        else:
            strategy = tf.distribute.get_strategy()
            device = "CPU"
            print("ℹ CPU MODE")
    
    print(f"  Replicas: {strategy.num_replicas_in_sync}")
    return strategy, device, strategy.num_replicas_in_sync

strategy, device, num_replicas = detect_hardware()


[STEP 1] HARDWARE DETECTION
----------------------------------------------------------------------
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
✓ GPU DETECTED - 2 device(s)
  Mixed precision: Enabled
  Replicas: 2


I0000 00:00:1771235829.659315      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1771235829.665031      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [3]:
print("\n[STEP 2] DATA PREPARATION")
print("-"*70)

(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

y_train = keras.utils.to_categorical(y_train, 10)
y_test = keras.utils.to_categorical(y_test, 10)

print(f"  Training: {len(x_train):,} samples")
print(f"  Testing: {len(x_test):,} samples")


[STEP 2] DATA PREPARATION
----------------------------------------------------------------------
170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
  Training: 50,000 samples
  Testing: 10,000 samples


In [4]:
print("\n[STEP 3] DATA PIPELINE WITH AUGMENTATION")
print("-"*70)

BASE_BATCH = 128
GLOBAL_BATCH = BASE_BATCH * num_replicas
EPOCHS = 50

print(f"  Base batch: {BASE_BATCH}")
print(f"  Global batch: {GLOBAL_BATCH}")
print(f"  Epochs: {EPOCHS}")

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.2),
])

def augment(images, labels):
    return data_augmentation(images, training=True), labels

train_data = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_data = (train_data
              .shuffle(50000)
              .batch(GLOBAL_BATCH)
              .map(augment, num_parallel_calls=tf.data.AUTOTUNE)
              .cache()
              .prefetch(tf.data.AUTOTUNE))

test_data = tf.data.Dataset.from_tensor_slices((x_test, y_test))
test_data = test_data.batch(GLOBAL_BATCH).cache().prefetch(tf.data.AUTOTUNE)

print("  ✓ Augmentation: flip, rotation, zoom, contrast")
print("  ✓ Pipeline: shuffle → batch → augment → cache → prefetch")


[STEP 3] DATA PIPELINE WITH AUGMENTATION
----------------------------------------------------------------------
  Base batch: 128
  Global batch: 256
  Epochs: 50
  ✓ Augmentation: flip, rotation, zoom, contrast
  ✓ Pipeline: shuffle → batch → augment → cache → prefetch


In [5]:
print("\n[STEP 4] BUILDING ENHANCED CNN")
print("-"*70)

with strategy.scope():
    model = keras.Sequential([
        layers.Input(shape=(32, 32, 3)),
        
        layers.Conv2D(64, 3, padding='same', kernel_initializer='he_normal'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(64, 3, padding='same', kernel_initializer='he_normal'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(2),
        layers.Dropout(0.2),
        
        layers.Conv2D(128, 3, padding='same', kernel_initializer='he_normal'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(128, 3, padding='same', kernel_initializer='he_normal'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(2),
        layers.Dropout(0.3),
        
        layers.Conv2D(256, 3, padding='same', kernel_initializer='he_normal'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(256, 3, padding='same', kernel_initializer='he_normal'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(2),
        layers.Dropout(0.4),
        
        layers.Flatten(),
        layers.Dense(512, kernel_initializer='he_normal'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax', dtype='float32')
    ])
    
    initial_lr = 0.001
    optimizer = keras.optimizers.Adam(learning_rate=initial_lr)
    
    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

print(f"  Parameters: {model.count_params():,}")


[STEP 4] BUILDING ENHANCED CNN
----------------------------------------------------------------------
  Parameters: 3,253,834


In [6]:
print("\n[STEP 5] CONFIGURING CALLBACKS")
print("-"*70)

callbacks = [
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        'best_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=0
    )
]

print("  ✓ Learning rate reduction on plateau")
print("  ✓ Early stopping with patience=8")
print("  ✓ Model checkpointing")



[STEP 5] CONFIGURING CALLBACKS
----------------------------------------------------------------------
  ✓ Learning rate reduction on plateau
  ✓ Early stopping with patience=8
  ✓ Model checkpointing


In [7]:
print("\n[STEP 6] TRAINING")
print("="*70)
print(f"Device: {device} | Replicas: {num_replicas} | Batch: {GLOBAL_BATCH}")
print("-"*70)

start_time = time.time()

history = model.fit(
    train_data,
    epochs=EPOCHS,
    validation_data=test_data,
    callbacks=callbacks,
    verbose=1
)

train_time = time.time() - start_time


[STEP 6] TRAINING
Device: GPU | Replicas: 2 | Batch: 256
----------------------------------------------------------------------
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:

I0000 00:00:1771235846.426271     124 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1771235847.013351     123 cuda_dnn.cc:529] Loaded cuDNN version 91002


196/196 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.3114 - loss: 2.0588INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).


196/196 ━━━━━━━━━━━━━━━━━━━━ 35s 95ms/step - accuracy: 0.3117 - loss: 2.0575 - val_accuracy: 0.3779 - val_loss: 1.8041 - learning_rate: 0.0010
Epoch 2/50
195/196 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.4681 - loss: 1.4819

196/196 ━━━━━━━━━━━━━━━━━━━━ 11s 55ms/step - accuracy: 0.4683 - loss: 1.4813 - val_accuracy: 0.4843 - val_loss: 1.5864 - learning_rate: 0.0010
Epoch 3/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.5432 - loss: 1.2714

...

196/196 ━━━━━━━━━━━━━━━━━━━━ 11s 56ms/step - accuracy: 0.8497 - loss: 0.4178 - val_accuracy: 0.7754 - val_loss: 0.7274 - learning_rate: 5.0000e-04
Epoch 19/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 11s 55ms/step - accuracy: 0.8636 - loss: 0.3863 - val_accuracy: 0.7665 - val_loss: 0.7949 - learning_rate: 5.0000e-04
Epoch 20/50
195/196 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.8675 - loss: 0.3721

196/196 ━━━━━━━━━━━━━━━━━━━━ 12s 56ms/step - accuracy: 0.8675 - loss: 0.3720 - val_accuracy: 0.7778 - val_loss: 0.7478 - learning_rate: 5.0000e-04
Epoch 21/50
195/196 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.8735 - loss: 0.3510
Epoch 21: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
196/196 ━━━━━━━━━━━━━━━━━━━━ 11s 55ms/step - accuracy: 0.8736 - loss: 0.3509 - val_accuracy: 0.7628 - val_loss: 0.8368 - learning_rate: 5.0000e-04
Epoch 22/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.8843 - loss: 0.3200

196/196 ━━━━━━━━━━━━━━━━━━━━ 12s 56ms/step - accuracy: 0.8844 - loss: 0.3199 - val_accuracy: 0.7783 - val_loss: 0.7625 - learning_rate: 2.5000e-04
Epoch 23/50
195/196 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.8952 - loss: 0.2929

196/196 ━━━━━━━━━━━━━━━━━━━━ 12s 56ms/step - accuracy: 0.8953 - loss: 0.2928 - val_accuracy: 0.7918 - val_loss: 0.7175 - learning_rate: 2.5000e-04
Epoch 24/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 11s 55ms/step - accuracy: 0.9022 - loss: 0.2773 - val_accuracy: 0.7830 - val_loss: 0.7665 - learning_rate: 2.5000e-04
Epoch 25/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 11s 55ms/step - accuracy: 0.9069 - loss: 0.2614 - val_accuracy: 0.7831 - val_loss: 0.7680 - learning_rate: 2.5000e-04
Epoch 26/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.9089 - loss: 0.2515
Epoch 26: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.


196/196 ━━━━━━━━━━━━━━━━━━━━ 12s 56ms/step - accuracy: 0.9089 - loss: 0.2515 - val_accuracy: 0.7928 - val_loss: 0.7256 - learning_rate: 2.5000e-04
Epoch 27/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 11s 55ms/step - accuracy: 0.9135 - loss: 0.2428 - val_accuracy: 0.7901 - val_loss: 0.7455 - learning_rate: 1.2500e-04
Epoch 28/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 11s 55ms/step - accuracy: 0.9179 - loss: 0.2290 - val_accuracy: 0.7917 - val_loss: 0.7398 - learning_rate: 1.2500e-04
Epoch 29/50
195/196 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.9210 - loss: 0.2198
Epoch 29: ReduceLROnPlateau reducing learning rate to 6.25000029685907e-05.
196/196 ━━━━━━━━━━━━━━━━━━━━ 11s 55ms/step - accuracy: 0.9211 - loss: 0.2197 - val_accuracy: 0.7842 - val_loss: 0.7799 - learning_rate: 1.2500e-04
Epoch 30/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 12s 57ms/step - accuracy: 0.9243 - loss: 0.2126 - val_accuracy: 0.7924 - val_loss: 0.7592 - learning_rate: 6.2500e-05
Epoch 31/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 11s 52ms/step - accuracy

In [8]:
print("\n[STEP 7] FINAL EVALUATION")
print("="*70)

test_loss, test_acc = model.evaluate(test_data, verbose=0)

epochs_trained = len(history.history['loss'])
best_val_acc = max(history.history['val_accuracy'])

print(f"\n{'Metric':<30} {'Value':<20}")
print("-"*70)
print(f"{'Device':<30} {device}")
print(f"{'Replicas':<30} {num_replicas}")
print(f"{'Global Batch Size':<30} {GLOBAL_BATCH}")
print(f"{'Epochs Trained':<30} {epochs_trained}/{EPOCHS}")
print(f"{'Training Time':<30} {train_time:.2f} sec ({train_time/60:.2f} min)")
print(f"{'Time per Epoch':<30} {train_time/epochs_trained:.2f} sec")
print(f"{'Best Validation Accuracy':<30} {best_val_acc*100:.2f}%")
print(f"{'Final Test Accuracy':<30} {test_acc*100:.2f}%")
print(f"{'Test Loss':<30} {test_loss:.4f}")
print(f"{'Throughput':<30} {len(x_train)*epochs_trained/train_time:.0f} samples/sec")


[STEP 7] FINAL EVALUATION

Metric                         Value               
----------------------------------------------------------------------
Device                         GPU
Replicas                       2
Global Batch Size              256
Epochs Trained                 34/50
Training Time                  411.19 sec (6.85 min)
Time per Epoch                 12.09 sec
Best Validation Accuracy       79.28%
Final Test Accuracy            79.28%
Test Loss                      0.7255
Throughput                     4134 samples/sec


In [9]:
print("\n[STEP 8] TRAINING PROGRESS")
print("="*70)

print("\nAccuracy Over Epochs:")
for i in range(0, len(history.history['accuracy']), max(1, len(history.history['accuracy'])//10)):
    epoch = i + 1
    train_acc = history.history['accuracy'][i] * 100
    val_acc = history.history['val_accuracy'][i] * 100
    bar_length = int(val_acc / 2)
    bar = "█" * bar_length
    print(f"Epoch {epoch:3d}: Train {train_acc:5.2f}% | Val {val_acc:5.2f}% {bar}")


[STEP 8] TRAINING PROGRESS

Accuracy Over Epochs:
Epoch   1: Train 37.45% | Val 37.79% ██████████████████
Epoch   4: Train 61.05% | Val 58.02% █████████████████████████████
Epoch   7: Train 69.17% | Val 63.19% ███████████████████████████████
Epoch  10: Train 74.75% | Val 68.79% ██████████████████████████████████
Epoch  13: Train 78.58% | Val 69.98% ██████████████████████████████████
Epoch  16: Train 83.59% | Val 75.14% █████████████████████████████████████
Epoch  19: Train 86.56% | Val 76.65% ██████████████████████████████████████
Epoch  22: Train 89.07% | Val 77.83% ██████████████████████████████████████
Epoch  25: Train 90.95% | Val 78.31% ███████████████████████████████████████
Epoch  28: Train 92.25% | Val 79.17% ███████████████████████████████████████
Epoch  31: Train 64.11% | Val 10.00% █████
Epoch  34: Train 10.00% | Val 10.00% █████


In [10]:
print("="*70)
print("✓ TRAINING COMPLETED SUCCESSFULLY")
print("="*70)

if device == "GPU":
    print("\n[GPU INFORMATION]")
    print("-"*70)
    import os
    os.system('nvidia-smi --query-gpu=name,memory.total,memory.used,utilization.gpu --format=csv,noheader')
    print("-"*70)

✓ TRAINING COMPLETED SUCCESSFULLY

[GPU INFORMATION]
----------------------------------------------------------------------
Tesla T4, 15360 MiB, 2397 MiB, 4 %
Tesla T4, 15360 MiB, 2393 MiB, 17 %
----------------------------------------------------------------------
